# MuTech SLR: Interactive Widget Dashboard
## Graphical UI inside Jupyter Notebook for Phases 1 to 5
This notebook provides an interactive graphical control panel powered by `ipywidgets` and `pipeline_engine.py`.

### Features:
* **Interactive Run Buttons:** Execute each phase independently or run all phases end-to-end.
* **Live Validation Gates:** Automated checks display pass/fail status directly in the notebook output area.
* **Dynamic Controls:** Adjust the target number of topics ($k$) and toggle Rayyan/Rule-based screening mode with one click.


In [ ]:
import os
import sys
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Add parent directory to path
THIS_DIR = os.getcwd()
if THIS_DIR not in sys.path:
    sys.path.insert(0, THIS_DIR)

from src import pipeline_engine as pe

RAW_DIR = os.path.join(THIS_DIR, "data", "01_raw")
RAYYAN_DIR = os.path.join(THIS_DIR, "data", "02_screening")
OUT_DIR = os.path.join(THIS_DIR, "outputs")
DOCS_DIR = os.path.join(os.path.dirname(THIS_DIR), "docs")

# Pipeline State
state = {"p1": None, "p2": None, "p3": None, "p4": None, "p5": None}

# Widgets
out_area = widgets.Output()
status_area = widgets.Output()

chk_rayyan = widgets.Checkbox(value=True, description="Use Rayyan Screened Corpus (N=365)", indent=False)
slider_k = widgets.IntSlider(value=7, min=3, max=12, step=1, description="Target k:", continuous_update=False)

btn_p1 = widgets.Button(description="▶ Run Phase 1 (Ingest & Dedupe)", button_style="info", layout=widgets.Layout(width="250px"))
btn_p2 = widgets.Button(description="▶ Run Phase 2 (PRISMA Screen)", button_style="info", layout=widgets.Layout(width="250px"))
btn_p3 = widgets.Button(description="▶ Run Phase 3 (NLP Text Mining)", button_style="info", layout=widgets.Layout(width="250px"))
btn_p4 = widgets.Button(description="▶ Run Phase 4 (4-Model Suite)", button_style="info", layout=widgets.Layout(width="250px"))
btn_p5 = widgets.Button(description="▶ Run Phase 5 (SNA Network)", button_style="info", layout=widgets.Layout(width="250px"))
btn_all = widgets.Button(description="🚀 Run All Phases (1 to 5)", button_style="success", layout=widgets.Layout(width="250px"))

def on_p1_clicked(b):
    with out_area:
        print("[Phase 1] Starting ingestion and deduplication...")
        try:
            res = pe.run_phase1(RAW_DIR, OUT_DIR)
            state["p1"] = res
            print(f"✅ Gate 1 Passed: {res['total_identified']} raw -> {res['unique_records']} unique ({res['total_duplicates']} duplicates removed).")
        except Exception as e:
            print(f"❌ Error in Phase 1: {e}")

def on_p2_clicked(b):
    with out_area:
        if state["p1"] is None:
            on_p1_clicked(None)
        print("[Phase 2] Starting PRISMA screening...")
        try:
            res = pe.run_phase2(state["p1"]["data"], RAW_DIR, RAYYAN_DIR, OUT_DIR, use_rayyan=chk_rayyan.value)
            state["p2"] = res
            print(f"✅ Gate 2 Passed: {res['screened_count']} studies included ({res['mode']}).")
        except Exception as e:
            print(f"❌ Error in Phase 2: {e}")

def on_p3_clicked(b):
    with out_area:
        if state["p2"] is None:
            on_p2_clicked(None)
        print("[Phase 3] Starting NLP text mining and LSA...")
        try:
            res = pe.run_phase3(state["p2"]["data"], OUT_DIR)
            state["p3"] = res
            print(f"✅ Gate 3 Passed: {res['num_docs']} docs, {res['vocab_size']} terms, 35-D LSA space created.")
        except Exception as e:
            print(f"❌ Error in Phase 3: {e}")

def on_p4_clicked(b):
    with out_area:
        if state["p3"] is None:
            on_p3_clicked(None)
        print(f"[Phase 4] Starting 4-model benchmarking (k={slider_k.value})...")
        try:
            res = pe.run_phase4(state["p3"]["data"], state["p3"]["tfidf_matrix"], state["p3"]["lsa_matrix"], OUT_DIR, target_k=slider_k.value)
            state["p4"] = res
            print("✅ Gate 4 Passed: Model benchmarking complete.")
            df_comp = pd.DataFrame(res["comparison_table"])
            display(df_comp)
        except Exception as e:
            print(f"❌ Error in Phase 4: {e}")

def on_p5_clicked(b):
    with out_area:
        if state["p4"] is None:
            on_p4_clicked(None)
        print("[Phase 5] Starting SNA network generation...")
        try:
            res = pe.run_phase5(state["p4"]["data"], state["p3"]["lsa_matrix"], state["p4"]["km_labels"], OUT_DIR, DOCS_DIR)
            state["p5"] = res
            print(f"✅ Gate 5 Passed: Graph built ({res['num_nodes']} nodes, {res['num_edges']} edges, modularity: {res['modularity']}).")
        except Exception as e:
            print(f"❌ Error in Phase 5: {e}")

def on_all_clicked(b):
    with out_area:
        clear_output()
        print("🚀 Starting Full SLR Pipeline Execution...")
        on_p1_clicked(None)
        on_p2_clicked(None)
        on_p3_clicked(None)
        on_p4_clicked(None)
        on_p5_clicked(None)
        print("🎉 All 5 phases executed and validated successfully!")

btn_p1.on_click(on_p1_clicked)
btn_p2.on_click(on_p2_clicked)
btn_p3.on_click(on_p3_clicked)
btn_p4.on_click(on_p4_clicked)
btn_p5.on_click(on_p5_clicked)
btn_all.on_click(on_all_clicked)

# Display UI Layout
panel = widgets.VBox([
    widgets.HTML("<h3>🎛️ MuTech SLR Interactive Phase Controller</h3>"),
    widgets.HBox([chk_rayyan, slider_k]),
    widgets.HBox([btn_p1, btn_p2, btn_p3]),
    widgets.HBox([btn_p4, btn_p5, btn_all]),
    widgets.HTML("<hr/>"),
    out_area
])

display(panel)
